# TD 1 : De la donnée brute à la donnée exploitable : nettoyage, modélisation et statistiques descriptives multimodales

> **À rendre :** un résumé de ce TD (2 pages maximum), à déposer dans votre dépôt GitLab.

## Objectif de la séance

Ce TD suit un **fil rouge unique** :

> **Donnée brute → donnée nettoyée → données décrites → features → sélection de variables → représentation du texte.**

À la fin du TD, vous devez être capables de :

1. auditer et nettoyer un jeu de données réel ;
2. identifier le type et le rôle des variables ;
3. construire un jeu de données exploitable par un modèle (`X` et `y`) ;
4. produire et interpréter des statistiques descriptives pour des variables numériques et catégorielles ;
5. créer de nouvelles variables par *feature engineering* ;
6. comparer plusieurs approches de *feature selection* ;
7. décrire une variable textuelle et comprendre comment la transformer en représentation numérique.

**Organisation pédagogique :**
- **Concept** : notion à retenir ;
- **Démo** : code guidé ;
- **À vous** : exercice à compléter.

Les sections 1 à 6 constituent ce TD. La suite (transformation du texte en TF-IDF, puis ouverture vers l'image et l'audio) se trouve dans `TD1_suite.ipynb`.

## Jeux de données

| Modalité | Jeu de données | Utilisation |
|---|---|---|
| 🏠 Tabulaire | DVF : transactions immobilières à La Rochelle en 2024 | nettoyage, transformation, modélisation |
| 📝 Texte | Amazon Fine Food Reviews : échantillon de 10 000 avis | statistiques, feature engineering, texte |
| 🖼️ Image | `sklearn.datasets.load_digits` | représentation d'une image par des nombres |
| 🔊 Audio | Free Spoken Digit Dataset : échantillon de 60 fichiers | signal, durée, spectrogramme |

Le fichier DVF est utilisé au début parce qu'il ressemble à une **donnée brute issue d'un système réel**. Le jeu Amazon devient ensuite le **fil rouge principal** pour les statistiques, les features et le texte.

> Les chemins supposent que le dossier `data/` se trouve au même niveau que le notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 100)

DATA = "../data"

print("Bibliothèques chargées.")
print("Dossier de données :", DATA)


<a id="sec1"></a>
# 1. De la donnée brute à la donnée exploitable : nettoyage et transformation

### Concept

Avant toute analyse, il faut répondre à quatre questions :

1. **Quelle est la structure du fichier ?**
2. **Les lignes représentent-elles réellement les observations que je crois ?**
3. **Les types sont-ils corrects ?**
4. **Les valeurs et catégories sont-elles cohérentes ?**

Le nettoyage n'est donc pas seulement « supprimer les `NaN` ». Il consiste à comprendre la structure des données puis à appliquer les transformations nécessaires.


In [ ]:
dvf = pd.read_csv(f"{DATA}/dvf_la_rochelle_2024.csv")

print("Dimensions :", dvf.shape)
dvf.head(5)


### À vous : premier audit

Explorez les **colonnes** et les **lignes**. Identifiez au moins trois éléments qui méritent une vérification.

Utilisez notamment :

- `shape`
- `columns`
- `head()`
- `tail()`
- `info()`
- `describe(include="all")`

Regardez en particulier la transaction `2024-176717`.


In [ ]:
# TODO : premier audit



### Doublons et structure des observations

Dans DVF, plusieurs lignes peuvent correspondre à une même transaction (`id_mutation`) parce qu'une transaction peut concerner plusieurs lots ou composants.

Avant de supprimer des doublons, il faut donc comprendre ce que représente une ligne.


### À vous : doublons

1. Combien de lignes sont strictement identiques sur toutes les colonnes ?
2. Combien d’`id_mutation` apparaissent plusieurs fois dans le fichier ?
3. Combien d’`id_mutation` différents y a-t-il dans le fichier ?

Comparez les trois nombres et expliquez pourquoi ils sont différents.


In [ ]:
# TODO : lignes strictement identiques
nb_doublons_stricts = ...
print("Doublons strictement identiques :", nb_doublons_stricts)

# TODO : nombre de lignes appartenant à une transaction répétée
nb_lignes_transactions_repetes = ...
print("Lignes avec id_mutation répété :", nb_lignes_transactions_repetes)

# TODO : nombre de transactions distinctes
nb_transactions = ...
print("Id mutation distinctes :", nb_transactions)


### Transformer les catégories

Une variable catégorielle est une variable dont les valeurs représentent des catégories ou des groupes, plutôt que des quantités numériques.

Exemples :

- Sexe → Homme, Femme
- Ville → Paris, Lyon, Bordeaux
- Type de logement → Appartement, Maison
- Couleur → Rouge, Bleu, Vert
- Statut → Étudiant, Salarié, Retraité

On distingue généralement :

- **Catégorielle nominale** : pas d'ordre entre les catégories.
  → Ville = Paris, Lyon, Bordeaux
- **Catégorielle ordinale** : les catégories ont un ordre.
  → Satisfaction = Faible, Moyenne, Élevée



Les variables catégorielles sont souvent stockées sous forme de texte (`Maison`, `Appartement`, etc.). Un modèle statistique ou de machine learning travaille généralement avec une représentation numérique.

Le **One-Hot Encoding** est une méthode qui permet de transformer une variable catégorielle en plusieurs variables numériques binaires (0 ou 1).

Le **one-hot encoding** transforme par exemple :

| type_local | → | Appartement | Maison | Dépendance |
|---|---:|---:|---:|---:|
| Appartement | | 1 | 0 | 0 |
| Maison | | 0 | 1 | 0 |

Attention : encoder une catégorie ne signifie pas lui attribuer arbitrairement un ordre.


In [ ]:
# Démo : distribution de type_local
dvf["type_local"].value_counts(dropna=False)


In [ ]:
# Démo : one-hot encoding
type_local_encode = pd.get_dummies(
    dvf["type_local"].fillna("Non renseigne"),
    prefix="type_local",
    dtype=int
)
type_local_encode.head().style.hide(axis="index")


### À vous : transformer une autre variable

Encodez `nature_mutation` avec `pd.get_dummies()` puis vérifiez que les colonnes obtenues correspondent bien aux modalités observées avec `value_counts()`.

**Question :** pourquoi faut-il traiter explicitement les valeurs manquantes avant ou pendant l'encodage ?


In [ ]:
# TODO
nature_mutation_encode = ...
nature_mutation_encode.head()


### Créer une première feature

Une donnée exploitable n'est pas forcément une donnée brute. On peut construire une variable plus informative à partir de plusieurs colonnes.

Pour les logements, le **prix au m²** est plus comparable que le prix total :

`prix_m2 = valeur_fonciere / surface_reelle_bati`

Mais une feature calculée n'est jamais meilleure que les données dont elle dépend : une surface erronée produit un prix au m² erroné.


In [ ]:
batis = dvf[dvf["type_local"].isin(["Maison", "Appartement"])].copy()
batis["prix_m2"] = batis["valeur_fonciere"] / batis["surface_reelle_bati"]

batis["prix_m2"].describe()


### Question d'interprétation

Observez le minimum et le maximum de `prix_m2`.

- Voyez-vous des valeurs surprenantes ?
- Faut-il immédiatement les supprimer ?
- Quelle information faudrait-il vérifier avant de décider ?

**À retenir :** une anomalie doit d'abord être **diagnostiquée**, pas automatiquement supprimée.


<a id="sec2"></a>

# 2. Modéliser les données : préparer les variables

### Qu'est-ce que « modéliser les données » ?

**Modéliser les données**, ici, signifie simplement **préparer et représenter les données dans une forme adaptée à l'analyse que l'on souhaite réaliser**.

Les données brutes ne sont pas toujours directement utilisables. On peut par exemple :

* sélectionner les variables utiles ;
* traiter les valeurs manquantes ;
* transformer certaines variables ;
* transformer les catégories en nombres lorsque cela est nécessaire (le one-hot encoding, vu en section 1).

### Et `X` et `y` ?

Lorsque l'on souhaite faire une prédiction, on distingue généralement :

* **`y`** : ce que l'on cherche à prédire ;
* **`X`** : les variables utilisées pour faire cette prédiction.

Par exemple, si l'on cherche à prédire la **valeur foncière** :

```text
X = surface, nombre de pièces, type_local, ...
y = valeur foncière
```

Pour le moment, retenons simplement :

> **`X` = les informations que l'on utilise**
> **`y` = l'information que l'on cherche à prédire**

In [ ]:
df = pd.read_csv(f"{DATA}/amazon_reviews_sample.csv")

print("Dimensions :", df.shape)
df.head(3)


In [ ]:
# Types techniques détectés par pandas
df.dtypes


### À vous : reconnaître les types de variables

Dans cette partie, **on cherche à comprendre quelles informations contient chaque colonne et à déterminer quelles colonnes peuvent être utiles pour analyser les avis**.

Observez les différentes colonnes du jeu de données et indiquez **quel type d'information** contient chacune d'elles.

| Colonne       | Type de variable |
| ------------- | ---------------- |
| `ProductId`   | ???              |
| `Score`       | ???              |
| `Time`        | ???              |
| `Helpfulness` | ???              |
| `Text`        | ???              |

**Aide** : les principaux types de variables

* **Numérique** : contient une quantité ou une mesure.
  *Exemple :* prix, surface, âge, nombre de pièces.

* **Catégorielle** : contient des catégories ou des groupes.
  *Exemple :* maison/appartement, ville, couleur.

* **Temporelle** : contient une date ou un moment dans le temps.
  *Exemple :* date d'achat, année, heure.

* **Textuelle** : contient du texte libre.
  *Exemple :* commentaire, description, avis.

* **Identifiant** : sert principalement à identifier une observation ou un objet.
  *Exemple :* numéro de produit, numéro de client.

> **Attention :** le type affiché par Python ne correspond pas toujours au type statistique de la variable. Une colonne peut contenir des nombres sans représenter une quantité. Par exemple, `ProductId` peut être numérique dans Python, mais il s'agit avant tout d'un **identifiant**.

### Questions

1. Quel est le type de chaque colonne ?
2. Quelles colonnes peuvent apporter des **informations utiles pour analyser les avis** ?
3. Quelle colonne ressemble davantage à un **identifiant** qu'à une véritable caractéristique du produit ou de l'avis ?
4. `Time` peut-être enregistré sous la forme d'un nombre. Pourquoi ce nombre représente-t-il malgré tout une **information temporelle** ?



<a id="sec3"></a>

# 3. Statistiques descriptives : variables numériques et catégorielles

Avant de faire des prédictions, il est important de **comprendre les données que l'on possède**.

Les **statistiques descriptives** permettent de résumer les données avec quelques indicateurs et quelques graphiques.

## 3.1 Décrire une variable numérique

Une variable numérique contient des **nombres qui représentent une quantité** : un prix, une surface, un âge, un nombre de pièces, une note, etc.

Pour la décrire, on peut chercher plusieurs informations :

* **Tendance centrale** : indique où se situent généralement les valeurs.

  * **Moyenne** : somme des valeurs divisée par leur nombre.
  * **Médiane** : valeur qui partage les données en deux groupes de même taille.

* **Dispersion** : indique si les valeurs sont proches ou éloignées les unes des autres.

  * **Écart-type** : mesure à quel point les valeurs s'éloignent de la moyenne.
  * **Étendue** : différence entre la plus grande et la plus petite valeur.
  * **Variance** : autre mesure de la dispersion des valeurs.

* **Position** : permet de situer une valeur par rapport aux autres.

  * **Quantiles** : partagent les données en plusieurs groupes de même taille. Par exemple, le premier quartile indique la valeur en dessous de laquelle se trouvent 25 % des observations.

* **Forme de la distribution** : permet de voir comment les valeurs sont réparties.

  * **Histogramme** : montre la répartition des valeurs.
  * **Boxplot** : permet notamment de visualiser la médiane, la dispersion et les valeurs atypiques.
  * **Asymétrie** : indique si la distribution est équilibrée ou si elle est davantage étirée d'un côté.

## 3.2 Décrire une variable catégorielle

Une variable catégorielle contient des **catégories ou des groupes**, par exemple le type de logement (`Maison`, `Appartement`), une ville ou un type de produit.

Pour la décrire, on peut compter **combien d'observations appartiennent à chaque catégorie** :

* **Effectif** : nombre d'observations dans une catégorie.
* **Fréquence** : nombre d'observations rapporté à l'ensemble des observations.
* **Proportion** : part que représente une catégorie dans l'ensemble des données, souvent exprimée en pourcentage.
* **Modalité rare** : catégorie qui apparaît très peu souvent dans les données.

Par exemple, si un jeu de données contient 1 000 logements et que 600 sont des appartements, alors l'effectif des appartements est **600** et leur proportion est **60 %**.

> **À retenir :** les statistiques descriptives permettent donc de répondre à des questions simples :
> **Quelles sont les valeurs typiques ? Les données sont-elles dispersées ? Quelles catégories sont les plus fréquentes ? Y a-t-il des valeurs inhabituelles ?**



In [ ]:
print("Score, statistiques descriptives")
print("Moyenne :", round(df["Score"].mean(), 3))
print("Médiane :", df["Score"].median())
print("Écart-type :", round(df["Score"].std(), 3))

df["Score"].describe()


In [ ]:
# Distribution de Score
df["Score"].plot(kind="hist", bins=5, figsize=(6, 3.5))
plt.title("Distribution des notes")
plt.xlabel("Score")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot
df["Score"].plot(kind="box", figsize=(6, 2.5))
plt.title("Boxplot des notes")
plt.tight_layout()
plt.show()


### À vous : interpréter plutôt que calculer

1. La moyenne et la médiane sont-elles proches ?
2. La distribution paraît-elle équilibrée ?
3. Que vous apprend le boxplot ?
4. Quelle note est majoritaire ?

Calculez ensuite les fréquences et pourcentages de chaque note.


In [ ]:
# TODO : fréquences et proportions
frequences_score = ...
pourcentages_score = ...

print("Fréquences :")
print(frequences_score)

print("\nPourcentages :")
print(pourcentages_score)


### Décrire une variable numérique construite

Nous allons maintenant créer `TextLength`, puis appliquer exactement le même raisonnement :

> **construire → décrire → visualiser → interpréter**


In [ ]:
# TODO
df["TextLength"] = ...

print("Moyenne :", ...)
print("Médiane :", ...)
print("Écart-type :", ...)

# TODO : histogramme
...


### Relation entre deux variables

Lorsque l'on étudie deux variables numériques, on peut se demander si **elles évoluent ensemble**.

Par exemple, on peut chercher à savoir si une augmentation de la **surface d'un logement** est associée à une augmentation de son **prix**.

La **corrélation** est un indicateur qui permet de mesurer la **force et le sens d'une relation entre deux variables**.

* Une corrélation **positive** signifie que les deux variables ont tendance à augmenter ensemble.
* Une corrélation **négative** signifie que lorsque l'une augmente, l'autre a tendance à diminuer.
* Une corrélation **proche de zéro** signifie qu'on ne détecte pas de relation linéaire claire entre les deux variables.

### Deux mesures courantes de corrélation `Pearson` et `Spearman`


### Pearson : mesurer une relation linéaire

La **corrélation de Pearson** permet de mesurer si deux variables numériques ont tendance à **évoluer ensemble de manière linéaire**.

Par exemple, imaginons quelques logements :

| Surface (m²) | Prix (€) |
| -----------: | -------: |
|           40 |  120 000 |
|           50 |  150 000 |
|           60 |  180 000 |
|           70 |  210 000 |
|           80 |  240 000 |

On observe ici que **plus la surface augmente, plus le prix augmente**.

La corrélation de Pearson sera alors **proche de +1**, ce qui indique une **forte relation linéaire positive**.

À l'inverse :

* une corrélation proche de **+1** → les deux variables augmentent ensemble ;
* une corrélation proche de **-1** → quand une variable augmente, l'autre diminue ;
* une corrélation proche de **0** → pas de relation linéaire claire.

>Pearson mesure une **association**, et non une relation de cause à effet. Une forte corrélation ne signifie donc pas nécessairement qu'une variable **cause** l'autre.


### Spearman : comparer les rangs

La **corrélation de Spearman** cherche à savoir si, lorsque les valeurs d'une variable augmentent, les valeurs de l'autre variable ont tendance à **augmenter ou diminuer elles aussi**.

Elle ne regarde pas directement les valeurs, mais plutôt **leur classement (leur rang)**.

Par exemple, imaginons que l'on compare le **nombre d'heures d'étude** et la **note obtenue** :

| Étudiant | Heures d'étude | Note |
| -------- | -------------: | ---: |
| A        |              1 |    8 |
| B        |              2 |   10 |
| C        |              4 |   13 |
| D        |              7 |   15 |
| E        |             10 |   16 |

Même si la progression des notes n'est pas parfaitement régulière, on constate que :

**plus le nombre d'heures d'étude augmente, plus la note a tendance à augmenter.**

Les rangs suivent donc globalement le même ordre :

```text
Heures d'étude :  1 → 2 → 4 → 7 → 10
Note           :  8 → 10 → 13 → 15 → 16
```

La corrélation de **Spearman** sera alors **positive et élevée**.

### Pearson ou Spearman ?

* **Pearson** → cherche surtout une relation **linéaire** entre les valeurs.
* **Spearman** → regarde surtout si les **rangs évoluent dans le même sens**.

> **À retenir :** Pearson s'intéresse davantage aux **valeurs**, tandis que Spearman s'intéresse davantage à leur **ordre**.


In [ ]:
# 
from scipy.stats import pearsonr, spearmanr

pearson_r, _ = pearsonr(df["Score"], df["TextLength"])
spearman_r, _ = spearmanr(df["Score"], df["TextLength"])

print("Pearson  :", round(pearson_r, 3))
print("Spearman :", round(spearman_r, 3))


### À vous: comparaison de groupes

Comparez la longueur moyenne des avis notés **5** et **1**.

Cette comparaison confirme-t-elle, nuance-t-elle ou contredit-elle votre interprétation de la corrélation ?


In [ ]:
# TODO
moyenne_5 = ...
moyenne_1 = ...

print("Longueur moyenne (5 étoiles) :", moyenne_5)
print("Longueur moyenne (1 étoile)  :", moyenne_1)


<a id="sec4"></a>
# 4. De la donnée brute à la donnée modèle : feature engineering

### Concept

Le **feature engineering** consiste à créer des variables qui représentent mieux le phénomène étudié.

Exemples :

- une date → année, mois, jour de la semaine ;
- un texte → longueur, nombre de mots, vocabulaire ;
- deux variables → ratio ou différence ;
- plusieurs observations → nombre d'événements antérieurs.

L'objectif n'est pas de créer « le plus de variables possible », mais des variables **pertinentes, interprétables et disponibles au moment de la prédiction**.


In [ ]:
# Transformer le timestamp en variables temporelles
df["Time"] = pd.to_datetime(df["Time"], unit="s")

df["annee"] = df["Time"].dt.year
df["mois"] = df["Time"].dt.month
df["jour_semaine"] = df["Time"].dt.dayofweek
df["est_weekend"] = df["jour_semaine"] >= 5

df[["Time", "annee", "mois", "jour_semaine", "est_weekend"]].head()


### À vous : utiliser l'historique des avis

Un même produit peut recevoir plusieurs avis au cours du temps.

Pour chaque avis, nous voulons créer une nouvelle variable appelée **`nb_avis_avant`**.

Cette variable indique **combien d'avis sur le même produit ont été publiés avant la date de cet avis**.

Par exemple :

| ProductId | Date de l'avis | `nb_avis_avant` |
| --------- | -------------- | --------------: |
| A         | 1 janvier      |               0 |
| A         | 5 janvier      |               1 |
| A         | 10 janvier     |               2 |
| A         | 20 janvier     |               3 |
| B         | 3 janvier      |               0 |
| B         | 15 janvier     |               1 |

### À vous de jouer

Construisez la variable **`nb_avis_avant`** à partir de `ProductId` et `Time`.

**Indice :**

* Commencez par trier les avis par produit et par date.
* Puis cherchez une fonction permettant de **compter les avis précédents pour chaque produit**.

### Question

`nb_avis_avant` peut-il apporter **plus d'information** que `ProductId` seul ?




In [ ]:
df = df.sort_values(["ProductId", "Time"]).reset_index(drop=True)

# TODO
df["nb_avis_avant"] = ...

df[["ProductId", "Time", "nb_avis_avant"]].head(10)


### Éviter les fuites d'information

Une feature est valide si elle est **disponible au moment où la prédiction est faite**.

Par exemple, `nb_avis_avant` est construit avec les avis précédents. En revanche, utiliser le nombre **total d'avis du produit**, calculé avec tout le dataset, peut utiliser indirectement des informations futures.

> **Feature engineering = créer de l'information utile sans regarder le futur.**


<a id="sec5"></a>

# 5. Sélection de variables : quelles variables conserver ?

Après avoir créé plusieurs variables, nous pouvons avoir **plusieurs features**.

Toutes les variables ne sont pas forcément utiles. Nous allons donc chercher à identifier celles qui semblent apporter le plus d'information.

Il existe plusieurs familles de méthodes :

* **Méthode filtre** : utilise un critère statistique pour évaluer les variables.
* **Méthode wrapper** : teste différents ensembles de variables avec un modèle.
* **Méthode embarquée** : le modèle participe directement à la sélection.

Nous allons comparer les trois, sur le même jeu de données (Iris).

## 5.1 Méthode filtre

La méthode filtre consiste à **évaluer les variables avant d'utiliser un modèle**.

Pour chaque variable, nous calculons un score permettant de mesurer son lien avec la variable `y`.

Dans notre exemple, nous allons utiliser le **score ANOVA F**.

L'idée est simple :

* **score F élevé** → la variable présente des différences importantes entre les groupes de `y` ;
* **score F faible** → les valeurs sont plus proches entre les groupes.

Le score permet donc de **classer les variables** selon leur intérêt.


### Exemple

Supposons que `y` indique l'**espèce d'une fleur** (le dataset Iris).

Nous disposons de plusieurs variables numériques :

| Variable             | Information         |
| --------------------- | -------------------- |
| `sepal length (cm)`  | longueur du sépale   |
| `sepal width (cm)`   | largeur du sépale    |
| `petal length (cm)`  | longueur du pétale   |
| `petal width (cm)`   | largeur du pétale    |

Nous voulons savoir quelles variables présentent les différences les plus importantes entre les trois espèces.

L'ANOVA F va donner **un score à chaque variable**.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()

In [ ]:
import pandas as pd

df_iris = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df_iris["target"] = iris.target

df_iris.head()

Dans notre exemple :

* `X` contient les mesures des fleurs ;
* `y` indique l'espèce de la fleur.

Nous allons utiliser ANOVA F pour déterminer quelles variables de `X` sont les plus intéressantes pour distinguer les catégories de `y`.


In [ ]:
X = df_iris.drop(columns="target")

X.head()

In [ ]:
y = df_iris["target"]

y.head()

In [ ]:
# Observer les catégories de y
# On découvre qu'il existe 3 groupes.
y.value_counts()

In [ ]:
# Importer ANOVA F
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
# Créer le sélecteur
selecteur = SelectKBest(
    score_func=f_classif, #indique que nous utilisons ANOVA F.
    k="all"
)

In [ ]:
# Calculer les scores
selecteur.fit(X, y)

In [ ]:
#  Associer les scores aux noms
scores_filtre = pd.Series(
    selecteur.scores_,
    index=X.columns
)

scores_filtre

In [ ]:
# Classer les variables
scores_filtre = scores_filtre.sort_values(
    ascending=False
)

scores_filtre

### À vous : interpréter les scores

Observez le résultat obtenu.

1. Quelle variable possède le score ANOVA F le plus élevé ?
2. Quelle variable possède le score le plus faible ?
3. Quelles sont les deux variables qui semblent les plus intéressantes pour distinguer les espèces de fleurs ?

**Rappel :**

> Plus le score ANOVA F est élevé, plus les valeurs de la variable sont différentes entre les groupes de `y`.


In [ ]:
import matplotlib.pyplot as plt

scores_filtre.sort_values().plot(
    kind="barh",
    figsize=(8, 4)
)

plt.xlabel("Score ANOVA F")
plt.ylabel("Variables")
plt.title("Sélection des variables avec ANOVA F")
plt.show()

In [ ]:
# Je souhaite conserver les 2 variables ayant les meilleurs scores ANOVA F.
selecteur_2 = SelectKBest(
    score_func=f_classif,
    k=2
)

X_selection = selecteur_2.fit_transform(X, y)

X_selection.shape

In [ ]:
variables_selectionnees = X.columns[
    selecteur_2.get_support()
]

variables_selectionnees

In [ ]:
# Créer le nouveau X
X_reduit = X[variables_selectionnees]

X_reduit.head()

### À retenir

Nous avons suivi les étapes suivantes :

**1. Préparer les données**

```text
X → variables numériques
y → catégories à comparer
```

**2. Calculer un score ANOVA F**

Chaque variable reçoit un score.

**3. Classer les variables**

Les scores les plus élevés apparaissent en premier.

**4. Visualiser les résultats**

Un graphique permet de comparer facilement les variables.

**5. Sélectionner les meilleures variables**

Nous pouvons finalement conserver les variables ayant les meilleurs scores.

> **Méthode filtre :**
>
> **variables → score statistique → classement → sélection**


## 5.2 Méthode wrapper

La méthode wrapper consiste à **tester différents sous-ensembles de variables avec un modèle**, et à garder celui qui donne les meilleurs résultats.

**RFE, ce n'est pas un modèle.** C'est une technique qui utilise un modèle existant (ici une régression logistique) pour décider, étape par étape, quelle variable retirer. RFE répète ceci :

1. entraîner le modèle avec les variables actuellement en jeu (au tout début, c'est-à-dire toutes les variables) ;
2. regarder quelle variable a le moins pesé dans sa décision ;
3. retirer cette variable ;
4. recommencer, jusqu'à ce qu'il ne reste que le nombre de variables demandé.

Contrairement à la méthode filtre, elle prend en compte les **interactions entre les variables**, mais elle est plus coûteuse en calcul puisqu'elle réentraîne le modèle plusieurs fois.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

modele = LogisticRegression(max_iter=200)
rfe = RFE(modele, n_features_to_select=2)
rfe.fit(X, y)

variables_wrapper = X.columns[rfe.support_]
variables_wrapper

**À vous.** Comparez les variables gardées par RFE à celles qui avaient le meilleur score ANOVA F en 5.1. Sont-elles les mêmes ?

## 5.3 Méthode embarquée

La méthode embarquée laisse le **modèle participer directement à la sélection**, pendant son propre entraînement : pas besoin d'étape séparée, le modèle indique lui-même quelles variables il a utilisées.

Un exemple classique : le Random Forest (forêt aléatoire) indique, avec `feature_importances_`, l'importance de chaque variable dans ses décisions. Faisons-le ici sur Iris, pour comparer les trois méthodes sur le même jeu de données.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances

### Comparaison des trois méthodes

Les trois méthodes s'accordent sur le plus important : la longueur et la largeur du pétale ressortent en tête, partout.

En revanche, elles se contredisent sur les deux variables restantes. Le filtre et la méthode embarquée jugent la longueur du sépale plus utile que sa largeur ; le wrapper (RFE) fait l'inverse.

Ce n'est pas gênant ici (les deux variables du sépale comptent peu de toute façon), mais ça montre concrètement qu'il ne faut jamais se fier à une seule méthode sans vérifier : un accord entre méthodes est un bon signal, un désaccord doit pousser à creuser plutôt qu'à trancher au hasard.

<a id="sec6"></a>
# 6. Décrire une variable textuelle

Un texte libre n'a pas de moyenne ou d'écart-type directement interprétable.

On peut cependant construire des **descripteurs numériques** :

1. longueur en caractères ;
2. nombre de mots ;
3. nombre de mots distincts ;
4. fréquence des mots ;
5. représentation TF-IDF ;
6. éventuellement des représentations vectorielles plus avancées.

Le principe est toujours le même :

> **texte brut → nettoyage → représentation numérique → analyse/modèle**


### À vous : explorer le vocabulaire

Nous allons maintenant regarder **les mots utilisés dans les avis**.

Sur les **2 000 premiers avis** :

1. récupérez tous les mots présents dans les avis ;
2. supprimez les **stopwords anglais** ;
3. gardez uniquement les mots différents ;
4. comptez le nombre de mots différents.

### Questions

* Combien de mots avez-vous au total ?
* Combien de mots différents avez-vous après avoir retiré les stopwords ?
* Quelle différence observez-vous entre les deux nombres ?
* Quels sont les mots qui apparaissent le plus souvent ?

> **À retenir :**
>
> Le **nombre total de mots** compte toutes les occurrences.
>
> Le **vocabulaire** compte seulement les mots différents.
>
> Par exemple :
>
> `good good product product`
>
> contient **4 mots**, mais seulement **2 mots différents** : `good` et `product`.


### De la donnée textuelle à une représentation exploitable

Nous avons vu quels mots sont présents dans les avis et lesquels sont les plus fréquents.

Mais un ordinateur ne peut pas directement utiliser une phrase comme :

> `This product is very good`

pour faire des calculs.

Nous devons donc **transformer les textes en nombres**.

L'objectif est de représenter chaque avis sous une forme numérique, tout en gardant une information sur les mots qu'il contient.

Nous allons découvrir deux méthodes simples :

* **Bag of Words** : compter les mots présents dans chaque avis ;
* **TF-IDF** : donner plus d'importance aux mots fréquents dans un avis mais moins fréquents dans l'ensemble des avis.

> **Objectif :** passer de **texte brut** à une **représentation numérique** que l'on peut analyser et utiliser ensuite dans un modèle.


# Fin